# Route Distance and Slope Extraction

Build route-level distance and slope inputs using OSMnx, OpenRouteService/Open-Elevation style APIs, and stop coordinate data.

This notebook was renamed and output-stripped for repository publication. Original project data files from IETT are not included in this public repository.


In [ ]:
pip install osmnx networkx pandas requests

In [ ]:
import pandas as pd
import requests
import osmnx as ox
import networkx as nx
import os


# İstanbul'u kapsayan yol ağı
G = ox.graph_from_place("Istanbul, Turkey", network_type='drive')

# tek seferlik indirme
ox.save_graphml(G, "istanbul_drive.graphml")

print("İstanbul haritası indirildi.")


In [ ]:
import pandas as pd
import requests
import osmnx as ox
import networkx as nx
import os

# Open-Elevation API ile yükseklik verisi al
def get_elevation(lat, lon):
    url = "https://api.open-elevation.com/api/v1/lookup"
    response = requests.post(url, json={"locations": [{"latitude": lat, "longitude": lon}]})
    if response.status_code == 200:
        return response.json()["results"][0]["elevation"]
    return None

# Mesafe hesapla (önceden indirilen harita kullanılarak)
def get_route_distance(G, origin_point, destination_point):
    origin_node = ox.distance.nearest_nodes(G, origin_point[1], origin_point[0])
    destination_node = ox.distance.nearest_nodes(G, destination_point[1], destination_point[0])
    length = nx.shortest_path_length(G, origin_node, destination_node, weight='length')
    return length / 1000

# Harita ve veri dosyasını yükle
G = ox.load_graphml("istanbul_drive.graphml")
df = pd.read_csv("filtrelenmis_hatlar_kordinatlar_duzgun_utf8sig.csv")

# Hat 32 hariç tüm hat kodlarını al
hat_kodlari = df['Hat Kodu'].unique()
hat_kodlari = [kod for kod in hat_kodlari if str(kod) != '32']

# Elevation cache kontrol
elevation_cache_path = "durak_elevation_cache_open_elevation.csv"
if os.path.exists(elevation_cache_path):
    elevation_df = pd.read_csv(elevation_cache_path)
    elevation_dict = {(row['Y'], row['X']): row['Elevation'] for _, row in elevation_df.iterrows()}
else:
    elevation_dict = {}
    elevation_rows = []

results = []

for hat in hat_kodlari:
    line_stops = df[df['Hat Kodu'] == hat].reset_index(drop=True)

    # Yeni duraklar için elevation hesapla
    for _, row in line_stops.iterrows():
        key = (row['Y'], row['X'])
        if key not in elevation_dict:
            elevation = get_elevation(row['Y'], row['X'])
            elevation_dict[key] = elevation
            if not os.path.exists(elevation_cache_path):
                elevation_rows.append({'Durak Adı': row['Durak Adı'], 'Y': row['Y'], 'X': row['X'], 'Elevation': elevation})

    for i in range(len(line_stops) - 1):
        current_stop = line_stops.iloc[i]
        next_stop = line_stops.iloc[i + 1]
        origin_point = (current_stop['Y'], current_stop['X'])
        destination_point = (next_stop['Y'], next_stop['X'])

        try:
            distance_km = get_route_distance(G, origin_point, destination_point)
            elev_start = elevation_dict.get(origin_point)
            elev_end = elevation_dict.get(destination_point)

            if elev_start is None or elev_end is None:
                raise ValueError("Elevation verisi eksik")

            elev_diff = elev_end - elev_start
            slope_percent = (elev_diff / (distance_km * 1000)) * 100 if distance_km > 0 else 0

            results.append({
                "Hat Kodu": hat,
                "From": current_stop['Durak Adı'],
                "To": next_stop['Durak Adı'],
                "Route Distance (km)": round(distance_km, 3),
                "Elevation Start (m)": elev_start,
                "Elevation End (m)": elev_end,
                "Slope (%)": round(slope_percent, 2)
            })
        except Exception as e:
            results.append({
                "Hat Kodu": hat,
                "From": current_stop['Durak Adı'],
                "To": next_stop['Durak Adı'],
                "Route Distance (km)": None,
                "Elevation Start (m)": elevation_dict.get(origin_point),
                "Elevation End (m)": elevation_dict.get(destination_point),
                "Slope (%)": None,
                "Error": str(e)
            })

# Cache'e yaz (ilk defa oluşturuluyorsa)
if elevation_rows:
    pd.DataFrame(elevation_rows).to_csv(elevation_cache_path, index=False)


results_df = pd.DataFrame(results)
results_df.to_excel("tum_hatlar_32_haric_mesafe_egim.xlsx", index=False)
print("✅ Tüm hatlar (32 hariç) işlendi ve sonuçlar Excel dosyasına kaydedildi.")
